In [1]:
%reload_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.style = 'default'  # This is the key fix

In [2]:
# # First, close all SQL connections
# %sql --close sqlite:///practice3.db

In [3]:
# # Delete the database file
# !rm -f practice3.db

In [4]:
# Connect to an SQLite database (creates if doesn't exist)
%sql sqlite:///practice3.db

##### SQL Injection

SQL injection is a code injection technique that might destroy your database.

SQL injection is one of the most common web hacking techniques.

SQL injection is the placement of malicious code in SQL statements, via web page input.

SQL injection usually occurs when you ask a user for input, like their username/userid, and instead of a name/id, the user gives you an SQL statement that you will unknowingly run on your database.

Look at the following example which creates a SELECT statement by adding a variable (txtUserId) to a select string.

 The variable is fetched from user input (getRequestString):


txtUserId = getRequestString("UserId");
txtSQL = "SELECT * FROM Users WHERE UserId = " + txtUserId;


The original purpose of the code was to create an SQL statement to select a user, with a given user id.

If there is nothing to prevent a user from entering "wrong" input, the user can enter some "smart" input like this:

HTML Inputbox : UserId: 105 OR 1=1

Then, the SQL statement will look like this:

SELECT * FROM Users WHERE UserId = 105 OR 1=1;

The SQL above is valid and will return ALL rows from the "Users" table, since OR 1=1 is always TRUE.

Does the example above look dangerous? What if the "Users" table contains names and passwords?

The SQL statement above is much the same as this:

SELECT UserId, Name, Password FROM Users WHERE UserId = 105 or 1=1;


Use SQL Parameters for Protection:

To protect a web site from SQL injection, you can use SQL parameters.

SQL parameters are values that are added to an SQL query at execution time, in a controlled manner.

ASP.NET Razor Example
txtUserId = getRequestString("UserId");
txtSQL = "SELECT * FROM Users WHERE UserId = @0";
db.Execute(txtSQL,txtUserId);


### Advanced SQL 

In SQL, a window function or analytic function is a function which uses values from one or multiple rows to return a value for each row.
(This contrasts with an aggregate function, which returns a single value for multiple rows.)


Window functions have an OVER clause; any function without an OVER clause is not a window function, but rather an aggregate or single-row (scalar) function.

#### Rank | Dense Rank | Row Number

In [8]:
%%sql
create table if not exists Grade (Marks integer);

 * sqlite:///practice3.db


""


In [9]:
%sql select * from Grade;

 * sqlite:///practice3.db


""


In [ ]:
%sql SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:///practice3.db


""


In [14]:
%sql insert into Grade (Marks) values (78), (68), (68), (65), (85);

 * sqlite:///practice3.db


""


In [15]:
%sql select * from Grade;

 * sqlite:///practice3.db


,Marks
0,85
1,78
2,68
3,68
4,65
5,85


In [17]:
%%sql
select *,
rank() over(order by Marks desc) rnk,
dense_rank() over(order by Marks desc) dense_rnk,
row_number() over(order by Marks desc) row_number

from Grade; 

 * sqlite:///practice3.db


,Marks,rnk,dense_rnk,row_number
0,85,1,1,1
1,85,1,1,2
2,78,3,2,3
3,68,4,3,4
4,68,4,3,5
5,65,6,4,6


In [19]:
%%sql
create table if not exists Dept (ID integer primary key, Department text, Salary integer);

 * sqlite:///practice3.db


""


In [21]:
%%sql
insert into Dept values 
(1, 'Sales', 1000),
(2,'IT',1500),
(3,'Sales',2000),
(4,'Sales',1700),
(5, 'IT',1800),
(6,'Accounts',1200),
(7,'Accounts',1100);

 * sqlite:///practice3.db


""


In [22]:
%sql select * from Dept;

 * sqlite:///practice3.db


,ID,Department,Salary
0,1,Sales,1000
1,2,IT,1500
2,3,Sales,2000
3,4,Sales,1700
4,5,IT,1800
5,6,Accounts,1200
6,7,Accounts,1100


In [23]:
%%sql
select *, rank() over (partition by Department order by Salary desc) 
as emp_rank from Dept;

 * sqlite:///practice3.db


,ID,Department,Salary,emp_rank
0,6,Accounts,1200,1
1,7,Accounts,1100,2
2,5,IT,1800,1
3,2,IT,1500,2
4,3,Sales,2000,1
5,4,Sales,1700,2
6,1,Sales,1000,3


#### Rows Between clause in SQL

In [24]:
%%sql
create table if not exists DeptSales (ID integer primary key,
Date date, Sales integer);

 * sqlite:///practice3.db


""


In [25]:
%%sql
insert into DeptSales values
(1, '22-06-2022',603),
(2,'21-06-2022',478),
(3,'20-06-2022',679),
(4,'19-06-2022',443),
(5,'18-06-2022',540),
(6,'17-06-2022',740),
(7,'16-06-2022',850),
(8,'15-06-2022',604),
(9,'14-06-2022',339),
(10,'13-06-2022',905);

 * sqlite:///practice3.db


""


In [26]:
%sql select * from DeptSales;

 * sqlite:///practice3.db


,ID,Date,Sales
0,1,22-06-2022,603
1,2,21-06-2022,478
2,3,20-06-2022,679
3,4,19-06-2022,443
4,5,18-06-2022,540
5,6,17-06-2022,740
6,7,16-06-2022,850
7,8,15-06-2022,604
8,9,14-06-2022,339
9,10,13-06-2022,905


Data we want: current sales + prev. day sales+ next day sales for each row except first and last row


So, in general, we want current row value+ m rows values preceding + n rows values following.


In [27]:
%%sql
select *,Sum(Sales) over (order by Date rows between 1 preceding and 
1 following) as total_sales_today_yesterday_nextday from DeptSales;

 * sqlite:///practice3.db


,ID,Date,Sales,total_sales_today_yesterday_nextday
0,10,13-06-2022,905,1244
1,9,14-06-2022,339,1848
2,8,15-06-2022,604,1793
3,7,16-06-2022,850,2194
4,6,17-06-2022,740,2130
5,5,18-06-2022,540,1723
6,4,19-06-2022,443,1662
7,3,20-06-2022,679,1600
8,2,21-06-2022,478,1760
9,1,22-06-2022,603,1081


SQL Query for sum of all rows before a particular row and the all rows after a particular row.

In [28]:
%%sql
select *,Sum(Sales) over (order by Date rows between unbounded
preceding and unbounded following) as total_sales_before_after 
from DeptSales;

 * sqlite:///practice3.db


,ID,Date,Sales,total_sales_before_after
0,10,13-06-2022,905,6181
1,9,14-06-2022,339,6181
2,8,15-06-2022,604,6181
3,7,16-06-2022,850,6181
4,6,17-06-2022,740,6181
5,5,18-06-2022,540,6181
6,4,19-06-2022,443,6181
7,3,20-06-2022,679,6181
8,2,21-06-2022,478,6181
9,1,22-06-2022,603,6181


Cumulative Sum (Running Sum) in SQL

In [29]:
%%sql
select *,sum(Sales) over (order by Date rows between unbounded
preceding and current row) as running_sum from DeptSales;

 * sqlite:///practice3.db


,ID,Date,Sales,running_sum
0,10,13-06-2022,905,905
1,9,14-06-2022,339,1244
2,8,15-06-2022,604,1848
3,7,16-06-2022,850,2698
4,6,17-06-2022,740,3438
5,5,18-06-2022,540,3978
6,4,19-06-2022,443,4421
7,3,20-06-2022,679,5100
8,2,21-06-2022,478,5578
9,1,22-06-2022,603,6181


In [30]:
%%sql
create table if not exists StateSales(ID integer primary key,
State text, Date date, Sales);

 * sqlite:///practice3.db


""


In [31]:
%%sql
insert into StateSales values
(1, 'Jharkhand', '22-06-2022',603),
(2, 'Jharkhand', '12-06-2022',683),
(3, 'Jharjhand', '21-06-2022',693),
(4, 'Bihar', '22-06-2022',603),
(5, 'Bihar', '22-06-2022',603),
(6, 'Bihar', '22-06-2022',603),
(7, 'Maharastra', '22-06-2022',603),
(8, 'Maharastra', '22-06-2022',603),
(9, 'Maharastra', '22-06-2022',603)

 * sqlite:///practice3.db


""


In [32]:
%sql select * from StateSales

 * sqlite:///practice3.db


,ID,State,Date,Sales
0,1,Jharkhand,22-06-2022,603
1,2,Jharkhand,12-06-2022,683
2,3,Jharjhand,21-06-2022,693
3,4,Bihar,22-06-2022,603
4,5,Bihar,22-06-2022,603
5,6,Bihar,22-06-2022,603
6,7,Maharastra,22-06-2022,603
7,8,Maharastra,22-06-2022,603
8,9,Maharastra,22-06-2022,603


In [33]:
%%sql
select *,sum(Sales) over (partition by State order by Date rows 
between unbounded preceding and current row) as running_total_statewise
from StateSales;

 * sqlite:///practice3.db


,ID,State,Date,Sales,running_total_statewise
0,4,Bihar,22-06-2022,603,603
1,5,Bihar,22-06-2022,603,1206
2,6,Bihar,22-06-2022,603,1809
3,3,Jharjhand,21-06-2022,693,693
4,2,Jharkhand,12-06-2022,683,683
5,1,Jharkhand,22-06-2022,603,1286
6,7,Maharastra,22-06-2022,603,603
7,8,Maharastra,22-06-2022,603,1206
8,9,Maharastra,22-06-2022,603,1809


First Value, Last Value and Nth Value in SQL

In [37]:
%%sql
select *, first_value(Sales) over (partition by State order by Date) first_day_sales,
last_value(Sales) over (partition by State order by Date rows between unbounded preceding and unbounded following) last_day_sales 
from StateSales

 * sqlite:///practice3.db


,ID,State,Date,Sales,first_day_sales,last_day_sales
0,4,Bihar,22-06-2022,603,603,603
1,5,Bihar,22-06-2022,603,603,603
2,6,Bihar,22-06-2022,603,603,603
3,3,Jharjhand,21-06-2022,693,693,693
4,2,Jharkhand,12-06-2022,683,683,603
5,1,Jharkhand,22-06-2022,603,683,603
6,7,Maharastra,22-06-2022,603,603,603
7,8,Maharastra,22-06-2022,603,603,603
8,9,Maharastra,22-06-2022,603,603,603


In [38]:
%%sql
select *, first_value(Sales) over (partition by State order by Date) first_day_sales,
last_value(Sales) over (partition by State order by Date) last_day_sales 
from StateSales

 * sqlite:///practice3.db


,ID,State,Date,Sales,first_day_sales,last_day_sales
0,4,Bihar,22-06-2022,603,603,603
1,5,Bihar,22-06-2022,603,603,603
2,6,Bihar,22-06-2022,603,603,603
3,3,Jharjhand,21-06-2022,693,693,693
4,2,Jharkhand,12-06-2022,683,683,683
5,1,Jharkhand,22-06-2022,603,683,603
6,7,Maharastra,22-06-2022,603,603,603
7,8,Maharastra,22-06-2022,603,603,603
8,9,Maharastra,22-06-2022,603,603,603


In [42]:
%%sql
SELECT *,
       nth_value(Sales, 2) OVER (
           PARTITION BY State
           ORDER BY Date
           ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
       ) AS first_day_sales
FROM StateSales;

 * sqlite:///practice3.db


,ID,State,Date,Sales,first_day_sales
0,4,Bihar,22-06-2022,603,603.0
1,5,Bihar,22-06-2022,603,603.0
2,6,Bihar,22-06-2022,603,603.0
3,3,Jharjhand,21-06-2022,693,NaN
4,2,Jharkhand,12-06-2022,683,603.0
5,1,Jharkhand,22-06-2022,603,603.0
6,7,Maharastra,22-06-2022,603,603.0
7,8,Maharastra,22-06-2022,603,603.0
8,9,Maharastra,22-06-2022,603,603.0


By default, the window frame is:

`RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` 
      
So the nth value may not exist yet for early rows.


#### Moving Average/Rolling Average/Rolling Mean in SQL

it is a smoothing technique for time series data. it removed the spikes/noise


In [43]:
%%sql
create table if not exists MovingAvg(Date date,UnitSales integer)

 * sqlite:///practice3.db


""


In [44]:
%%sql
insert into MovingAvg values
('28-05-2022',45),
('29-05-2022',36),
('30-05-2022',39),
('31-05-2022',45),
('01-06-2022',44),
('02-06-2022',35),
('03-06-2022',48),
('04-06-2022',45),
('05-06-2022',30)

 * sqlite:///practice3.db


""


In [45]:
%%sql
select * from MovingAvg

 * sqlite:///practice3.db


,Date,UnitSales
0,28-05-2022,45
1,29-05-2022,36
2,30-05-2022,39
3,31-05-2022,45
4,01-06-2022,44
5,02-06-2022,35
6,03-06-2022,48
7,04-06-2022,45
8,05-06-2022,30


In [48]:
%%sql
select *,Avg(UnitSales) over (order by Date rows between 2 preceding and current row) as ThreeDaysMovingAvg
from MovingAvg


 * sqlite:///practice3.db


,Date,UnitSales,ThreeDaysMovingAvg
0,01-06-2022,44,44.000000
1,02-06-2022,35,39.500000
2,03-06-2022,48,42.333333
3,04-06-2022,45,42.666667
4,05-06-2022,30,41.000000
5,28-05-2022,45,40.000000
6,29-05-2022,36,37.000000
7,30-05-2022,39,40.000000
8,31-05-2022,45,40.000000


In [50]:
%%sql
select *,Avg(UnitSales) over (rows between 2 preceding and current row) as ThreeDaysMovingAvg,
Avg(UnitSales) over (rows between 4 preceding and current row) as FiveDaysMovingAvg
from MovingAvg


 * sqlite:///practice3.db


,Date,UnitSales,ThreeDaysMovingAvg,FiveDaysMovingAvg
0,28-05-2022,45,45.000000,45.00
1,29-05-2022,36,40.500000,40.50
2,30-05-2022,39,40.000000,40.00
3,31-05-2022,45,40.000000,41.25
4,01-06-2022,44,42.666667,41.80
5,02-06-2022,35,41.333333,39.80
6,03-06-2022,48,42.333333,42.20
7,04-06-2022,45,42.666667,43.40
8,05-06-2022,30,41.000000,40.40


In [52]:
%%sql
create table NewStocks(Date date, Close float);

 * sqlite:///practice3.db


""


In [53]:
%%sql
insert into NewStocks values
('28-05-2021',1103.5),
('31-05-2021',1125.65),
('01-06-2021',1100.9),
('02-06-2021',1124.05),
('03-06-2021',1120.7),
('04-06-2021',1128.7),
('07-06-2021',1111.1),
('08-06-2021',1114.45),
('09-06-2021',1158.35)

 * sqlite:///practice3.db


""


In [56]:
%%sql
select *,avg(Close) over (rows between 2 preceding and current row) as ThreeDaysMovingAvg
from NewStocks

 * sqlite:///practice3.db


,Date,Close,ThreeDaysMovingAvg
0,28-05-2021,1103.50,1103.500000
1,31-05-2021,1125.65,1114.575000
2,01-06-2021,1100.90,1110.016667
3,02-06-2021,1124.05,1116.866667
4,03-06-2021,1120.70,1115.216667
5,04-06-2021,1128.70,1124.483333
6,07-06-2021,1111.10,1120.166667
7,08-06-2021,1114.45,1118.083333
8,09-06-2021,1158.35,1127.966667


#### Lead and Lag in SQL